In [1]:
import numpy as np
import sympy as sp
import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display,IFrame,HTML,Image


In [16]:
url='https://www.janestreet.com/puzzles/robot-baseball-index/'
res = requests.get(url)
soup = BeautifulSoup(res.content, 'html.parser')
y =[text for text in soup.body.stripped_strings]
#display([(i,j) for i,j in enumerate(y)])
display(Markdown("### "+y[13]+"\n"+" \n ".join(y[16:58])))


### Robot Baseball
The Artificial Automaton Athletics Association (Quad-A) is at it again, to compete with
postseason baseball they are developing a Robot Baseball competition. Games are composed
of a series of independent 
 at-bats 
 in which the batter is trying to maximize expected
score and the pitcher is trying to minimize expected score. 
 An 
 at-bat 
 is a series of 
 pitches 
 with a running count of balls and strikes, both
starting at zero. For each pitch, the pitcher decides whether to throw a ball or strike,
and the batter decides whether to wait or swing; these decisions are made secretly and
simultaneously. The results of these choices are as follows. 
 If the pitcher throws a ball and the batter waits, the count of balls is incremented by 1. 
 If the pitcher throws a strike and the batter waits, the count of strikes is incremented by 1. 
 If the pitcher throws a ball and the batter swings, the count of strikes is incremented by 1. 
 If the pitcher throws a strike and the batter swings, with probability 
 p 
 the batter
hits a home run 
 1 
 and with probability 
 1-p 
 the count of strikes is incremented by 1. 
 An at-bat ends when either: 
 The count of balls reaches 4, in which case the batter receives 1 point. 
 The count of strikes reaches 3, in which case the batter receives 0 points. 
 The batter hits a home run, in which case the batter receives 4 points. 
 By varying the size of the strike zone, Quad-A can adjust the value 
 p 
 , the probability a
pitched strike that is swung at results in a home run. They have found that viewers are
most excited by at-bats that reach a 
 full count 
 , that is, the at-bats that reach the
state of three balls and two strikes. Let 
 q 
 be the probability of at-bats reaching full
count; 
 q 
 is dependent on 
 p 
 . Assume the batter and pitcher are both using optimal mixed
strategies and Quad-A has chosen the 
 p 
 that maximizes 
 q 
 . Find this 
 q 
 , the maximal
probability at-bats reach full count, to 
 ten decimal places 
 . 
 Robot fielding has a long way to go.

In [18]:
def solve_zero_sum_2x2(a11, a12, a21, a22):
    # Mixed-candidate (if interior)
    D = (a11 + a22) - (a12 + a21)
    if abs(D) > 1e-14:
        r = (a22 - a21) / D        # batter's prob of row1
        c = (a22 - a12) / D        # pitcher's prob of col1
        if 0.0 <= r <= 1.0 and 0.0 <= c <= 1.0:
            v = (a11*a22 - a12*a21) / D
            return v, r, c

    # Otherwise the value is at a pure saddle (maximin = minimax)
    row1_worst = min(a11, a12)
    row2_worst = min(a21, a22)
    if row1_worst >= row2_worst:
        # Batter chooses row1 (Wait) pure; pitcher best-responds by minimizing that row
        v = row1_worst
        c = 1.0 if a11 <= a12 else 0.0    # pitcher picks the column that minimizes row1
        r = 1.0
    else:
        v = row2_worst
        c = 1.0 if a21 <= a22 else 0.0
        r = 0.0
    return v, r, c


def q_of_p(p):
    # States: (b, s) with b in {0,1,2,3}, s in {0,1,2}. Terminals occur when b=4, s=3, or HR.
    V = {}                      # V[(b,s)] = equilibrium run value
    strat = {}                  # strat[(b,s)] = (r, c)  r=Prob(Wait), c=Prob(Ball)
    R = {}                      # R[(b,s)]   = Prob(ever hit (3,2) from (b,s)) under these strategies

    # Backward induction order: descendants (b+1,s) and (b,s+1) must be available first.
    for b in (3, 2, 1, 0):
        for s in (2, 1, 0):
            # Full-count node is (3,2) – still nonterminal for V, but for R we'll set R[3,2]=1 later.
            # Build the 2x2 payoff matrix entries (to the batter).
            # Rows: Wait (row1), Swing (row2); Cols: Ball (col1), Strike (col2).

            # Next-state values (with terminal handling):
            # (Wait, Ball): -> (b+1, s) unless b=3 then Walk=1
            if b == 3:
                a11 = 1.0
            else:
                a11 = V[(b+1, s)]

            # (Wait, Strike): -> (b, s+1) unless s=2 then Strikeout=0
            a12 = 0.0 if s == 2 else V[(b, s+1)]

            # (Swing, Ball): -> (b, s+1) unless s=2 then Strikeout=0
            a21 = 0.0 if s == 2 else V[(b, s+1)]

            # (Swing, Strike): HR with prob p (reward 4) else strike added
            if s == 2:
                # Next strike is strikeout; only HR yields runs
                a22 = 4.0 * p
            else:
                a22 = 4.0 * p + (1.0 - p) * V[(b, s+1)]

            v, r, c = solve_zero_sum_2x2(a11, a12, a21, a22)
            V[(b, s)] = v
            strat[(b, s)] = (r, c)

    # Now compute reach probability R of ever visiting (3,2) given those strategies.
    # By definition, if we are already at (3,2), R=1.
    for b in (3, 2, 1, 0):
        for s in (2, 1, 0):
            if (b, s) == (3, 2):
                R[(b, s)] = 1.0
                continue

            r, c = strat[(b, s)]          # r=Prob(Wait), c=Prob(Ball)
            # Outcome probabilities by action profile:
            # (Ball, Wait) -> prob c*r; next (b+1,s) unless b=3 -> walk terminal
            prob_BW = c * r
            # (Strike, Wait) -> prob (1-c)*r; next (b,s+1) unless s=2 -> K terminal
            prob_SWa = (1.0 - c) * r
            # (Ball, Swing) -> prob c*(1-r); next (b,s+1) unless s=2 -> K terminal
            prob_BSg = c * (1.0 - r)
            # (Strike, Swing) -> prob (1-c)*(1-r); next: HR w.p. p (terminal), else strike added
            prob_SS = (1.0 - c) * (1.0 - r)

            # Accumulate reach probability; terminal outcomes contribute 0, except reaching (3,2)
            total = 0.0
            # (Ball, Wait)
            if b != 3:  # not an immediate walk
                total += prob_BW * R[(b+1, s)]
            # (Strike, Wait)
            if s != 2:  # not an immediate strikeout
                total += prob_SWa * R[(b, s+1)]
            # (Ball, Swing)
            if s != 2:  # not an immediate strikeout
                total += prob_BSg * R[(b, s+1)]
            # (Strike, Swing)
            if s != 2:
                # With prob p it's a HR -> terminal (no extra chance to reach (3,2));
                # with prob 1-p we go to (b, s+1)
                total += prob_SS * (1.0 - p) * R[(b, s+1)]
            else:
                # s == 2: next strike is strikeout; only HR happens (terminal), contributes 0
                pass

            R[(b, s)] = total

    return R[(0, 0)]


def golden_section_max(f, a=0.0, b=1.0, tol=1e-12, max_iter=200):
    invphi = (np.sqrt(5) - 1) / 2      # 1/phi ≈ 0.618
    invphi2 = (3 - np.sqrt(5)) / 2     # 1/phi^2 ≈ 0.382

    c = a + invphi2 * (b - a)
    d = a + invphi * (b - a)
    fc = f(c)
    fd = f(d)

    for _ in range(max_iter):
        if abs(b - a) < tol:
            break
        if fc < fd:
            a = c
            c = d
            fc = fd
            d = a + invphi * (b - a)
            fd = f(d)
        else:
            b = d
            d = c
            fd = fc
            c = a + invphi2 * (b - a)
            fc = f(c)

    # best point in final bracket
    p_star = (a + b) / 2.0
    q_star = f(p_star)
    return p_star, q_star

def q_wrapper(p):
    # Clip to [0,1] to be safe in the optimizer's numeric arithmetic
    p = float(min(1.0, max(0.0, p)))
    return q_of_p(p)

p_star, q_star = golden_section_max(q_wrapper, 0.0, 1.0, tol=1e-13)
print("p*  =", f"{p_star:.12f}")
print("q*  =", f"{q_star:.10f}")


p*  = 0.226973229469
q*  = 0.2959679934


In [24]:
url='https://www.janestreet.com/puzzles/robot-baseball-solution/'
res = requests.get(url)
soup = BeautifulSoup(res.content, 'html.parser')
y =[text for text in soup.body.stripped_strings]
#display([(i,j) for i,j in enumerate(y)])
display(Markdown("### "+y[14]+"\n"+" \n ".join(y[16:30])))


### October 2025 : Solution
We were tasked with fine-tuning the strike zone of Robot Baseball to
maximize the probability of an at bat reaching a full count, and
finding this maximal probability. The most straightforward way to do
this was to work backwards from full count down to the start of an at
bat, for a given strike zone probability 
 p 
 , determine the Nash
equilibrium of pitcher’s probability to throw a strike and the
batter’s probability to swing. 
 An interesting symmetry to simplify the work: the solver might notice
that the outcome of a pitch is symmetric with respect to the pitcher’s
choice and the batter’s choice. This means the probability of the
pitcher throwing a strike and the probability of the batter swinging
will be the same in any scenario (both players want to choose a
probability that makes the opponent indifferent between their two
options, otherwise their move is exploitable and will reduce their
expected points (or for the pitcher increases the opponents expected
points)). 
 Once we work backwards to the starting count of zero balls and zero
strikes, we can compute the probability of reaching a full count and
then adjust 
 p 
 in order to maximize it. Happily we are asked for the
maximal height of the function, which is easier to estimate than the
location of that maximum (the 
 p 
 value that achieves it). Careful
calibration finds the maximum probability of reaching full count to be 
 0.2959679934… 
 (achieved at approximately 
 p 
 =0.2269732…). 
 Other stats of interest: optimal baseball pitchers have an impressive
68% strike out rate and admit 21% walks. Batters hit home runs at 11%
of plate appearances, and have a slash line of .139/.318/.556.